# Phase 4-A｜TSM Research-only：完整非消耗性執行前檢查

這份 Notebook 只執行 Agent 的 `ResearchOnlyRunner.probe()`。會驗證 GitHub 的 **正式簽核紀錄**、新 Snapshot manifest/footer/receipt、Core 身分、交易成本與時間保護，**不執行回測、不保留研究額度、不讀取 Holdout OHLCV**。

資料集：`TSM_DEV_1M_20240529_20240628_18350b3adba8e28a7e5e`，使用已完成九項資料 PASS 的不可變 Snapshot。

本次驗收唯一成功狀態：`COMPATIBLE_READY_TO_CONSUME` 且 `slot_consumed = false`。任何阻擋項目皆直接停止並顯示原因，不進入真實 SMA 回測。請用全新 Colab runtime 執行 Run All。


In [ ]:
from pathlib import Path
import json, subprocess, sys, re, os
from datetime import datetime
from zoneinfo import ZoneInfo

AGENT_REPO = "https://github.com/hh4832/strategy-research-agent.git"
AGENT_SHA = "982296bf4caf8d9d931406fb030ed62842b9266a"
CORE_SHA = "b16179a7cc077c36046574769a32c367fa476cae"
CONFIG = "configs/research_only/tsm_sma_5_20_smoke_2024_06_dev_18350b3adba8e28a7e5e.json"
AUTH_ID = "RA-IBKR_TSM-SMA520_DEV-18350b3adba8e28a7e5e"
EXPECTED_MANIFEST_SHA = "8b6fc838b72ae3015f6eb418c3a77115e53a86f185e05c2249e9dcd58192cfc5"
EXPECTED_LINEAGE_SHA = "3455f2b3d39f4fd05dda7ca9fed472c9ff0bcbf345f89f177e78eccbe5fd92a4"
SNAPSHOT = Path("/content/drive/MyDrive/00Quant_Research/data/development_snapshots/TSM_DEV_1M_20240529_20240628_18350b3adba8e28a7e5e")
REPORT_PARENT = Path("/content/drive/MyDrive/00Quant_Research/phase4a_nonconsuming_probes")
AGENT_ROOT = Path("/content/strategy-research-agent-probe")
assert sys.version_info >= (3,11)
assert re.fullmatch(r"[0-9a-f]{40}", AGENT_SHA) and re.fullmatch(r"[0-9a-f]{40}", CORE_SHA)
print("Mode: GOVERNANCE + METADATA-ONLY PROBE; NO BACKTEST; NO BATCH CONSUMPTION")



In [ ]:
from google.colab import drive
drive.mount("/content/drive")
assert SNAPSHOT.is_dir(), "Derived Snapshot not found; do not substitute original full-history source"
for name in ("manifest.json","lineage.json","tsm_development_1m.parquet","binding.json"):
    assert (SNAPSHOT / name).is_file(), "Missing derived artifact: " + name
assert not AGENT_ROOT.exists(), "Start a fresh Colab runtime; do not overwrite existing checkout"
subprocess.run(["git","clone","--quiet",AGENT_REPO,str(AGENT_ROOT)],check=True)
subprocess.run(["git","-C",str(AGENT_ROOT),"checkout","--quiet","--detach",AGENT_SHA],check=True)
assert subprocess.check_output(["git","-C",str(AGENT_ROOT),"rev-parse","HEAD"],text=True).strip()==AGENT_SHA
assert not subprocess.check_output(["git","-C",str(AGENT_ROOT),"status","--porcelain"],text=True).strip()
subprocess.run([sys.executable,"-m","pip","install","--quiet","market-research-core @ git+https://github.com/hh4832/market-research-core.git@"+CORE_SHA],check=True)
subprocess.run([sys.executable,"-m","pip","install","--quiet","-e",str(AGENT_ROOT)+"[research]"],check=True)
for path in (AGENT_ROOT / "src",AGENT_ROOT):
    if str(path) not in sys.path:sys.path.insert(0,str(path))
from strategy_agent.research_only.runner import ResearchOnlyRunner
from strategy_agent.governance.research_authorization import ResearchAuthorizationBook
from market_research_core.tsm_native import core_code_identity
import hashlib
def sha(path):
    h=hashlib.sha256()
    with open(path,"rb") as f:
        for chunk in iter(lambda:f.read(1024*1024),b""):h.update(chunk)
    return h.hexdigest()
assert sha(SNAPSHOT/"manifest.json")==EXPECTED_MANIFEST_SHA
assert sha(SNAPSHOT/"lineage.json")==EXPECTED_LINEAGE_SHA
identity=core_code_identity()
assert identity["core_git_sha"]==CORE_SHA and identity["core_tree_dirty"] is not True
print("Pinned GitHub versions and immutable derived manifest/lineage: PASS")



In [ ]:
cfg_path=AGENT_ROOT / CONFIG
assert cfg_path.is_file(), "Pinned config missing"
cfg=json.loads(cfg_path.read_text())
assert cfg["authorization_id"]==AUTH_ID
assert cfg["dataset"]["snapshot_id"]==SNAPSHOT.name
assert cfg["dataset"]["producer_commit_sha"]==CORE_SHA
assert cfg["snapshot_binding"]["manifest_sha256"]==EXPECTED_MANIFEST_SHA
assert cfg["snapshot_binding"]["snapshot_pin"]["lineage_sha256"]==EXPECTED_LINEAGE_SHA
book=ResearchAuthorizationBook(AGENT_ROOT/"governance/research_authorizations.jsonl",AGENT_ROOT/"governance/approvers.json")
auth=book.get(AUTH_ID)
assert auth is not None and auth["approved_by"]=="hh4832", "New research authorization absent or unsigned"
assert auth["dataset"]==cfg["dataset"] and auth["max_batches"]==1
uses_before=(AGENT_ROOT/"registry/research_authorization_uses.jsonl").read_bytes()
formal_before=(AGENT_ROOT/"governance/market_readiness.json").read_bytes()
auth_before=(AGENT_ROOT/"governance/research_authorizations.jsonl").read_bytes()
holdout_before=(AGENT_ROOT/"governance/holdout_locks.jsonl").read_bytes()
print("Signed research-only authorization: PASS; budget remains untouched")



In [ ]:
runner=ResearchOnlyRunner(root=AGENT_ROOT)
result=runner.probe(cfg_path,snapshot_root=SNAPSHOT)
assert result["slot_consumed"] is False
assert (AGENT_ROOT/"registry/research_authorization_uses.jsonl").read_bytes()==uses_before
assert (AGENT_ROOT/"governance/market_readiness.json").read_bytes()==formal_before
assert (AGENT_ROOT/"governance/research_authorizations.jsonl").read_bytes()==auth_before
assert (AGENT_ROOT/"governance/holdout_locks.jsonl").read_bytes()==holdout_before
print("AGENT NON-CONSUMING PROBE RESULT:")
print(json.dumps(result,indent=2,ensure_ascii=False,default=str))
stamp=datetime.now(ZoneInfo("Asia/Taipei")).strftime("%Y%m%d_%H%M%S")
report_dir=REPORT_PARENT/(stamp+"_"+AGENT_SHA[:12])
report_dir.mkdir(parents=True,exist_ok=False)
with (report_dir/"probe_result.json").open("x",encoding="utf-8") as f:
    json.dump(result,f,indent=2,ensure_ascii=False,default=str)
with (report_dir/"run_info.txt").open("x",encoding="utf-8") as f:
    f.write(f"timestamp_tz=Asia/Taipei\nagent_sha={AGENT_SHA}\ncore_sha={CORE_SHA}\n"
            f"snapshot_id={SNAPSHOT.name}\nconfig_path={CONFIG}\nauthorization_id={AUTH_ID}\n"
            f"slot_consumed=false\nstrategy_run=false\n")
print("Saved immutable report:",report_dir)
assert result["status"]=="COMPATIBLE_READY_TO_CONSUME" and not result["blockers"], (
    "STOP: Probe has blockers. No backtest was run and no batch was consumed."
)
print("COMPATIBLE_READY_TO_CONSUME — STOP HERE. A separate explicit run is required.")

